<a href="https://colab.research.google.com/github/AbdAlRahman-Odeh-99/Two_Phases_Simulation/blob/main/notebooks/multiclass_supervised_unbiased_adaptive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
from sklearn.datasets import make_blobs
import math
import numpy as np
from numba import jit

def generate_data(nsamples=1000,nclasses=2,nviews=10,seed=42,snrdb=9):
    sval = 10**(snrdb/20)
    rng = np.random.default_rng(seed=seed)
    mm = rng.random(size=(nclasses,nviews)) * sval # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    #mm = np.concatenate([-mm, mm],axis=0) #
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1] #(observations, means, labels)

@jit
def pred_linear_cla(x_observe,class_means):
    mean_tr = class_means.T # (v,nc)
    diff_mean_sq = mean_tr[:,:,None] - mean_tr[:,None,:] # (sel, nc, nc)
    pairwise_mean_avg = 0.5 * (mean_tr[:,:,None] + mean_tr[:,None,:])
    inner_prod = np.sum(
        (x_observe[:,None,None] - pairwise_mean_avg) * diff_mean_sq,
        axis=0) > 0 # bool(nc,nc)
    # mask the diagonal
    np.fill_diagonal(inner_prod,False)
    # prediction
    y_pred = np.argmax(np.sum(inner_prod,axis=1))
    return y_pred

def bool_array_2_int(bool_arr):
    return int("".join(map(str, bool_arr.astype(int))), 2)


def enumerate_subsets_fast(arr):
        arr = arr.astype(int)
        n = len(arr)
        # 1. Convert the array into a binary integer (bitmask)
        parent_mask = 0
        for i in range(n):
            if arr[i] == 1:
                parent_mask |= (1<<i)
        # 2. Enumerate all sub_masks natively
        sub = parent_mask
        out = []
        while True:
            # 3. Convert the integer back to an array
            sub_arr = [0] * n
            for i in range(n):
                if sub & (1<<i):
                    sub_arr[i] = 1
            if np.any(sub_arr) and not np.all(sub_arr==arr):
                out.append(np.array(sub_arr).astype("bool"))
            # Break after processing the empty set (0)
            if sub ==0:
                break

            # The bitwise jump to the next valid subset
            sub = (sub - 1) & parent_mask
        return np.array(out)


In [9]:
import os
import sys
import time
import pandas as pd
import numpy as np

def reliability_learn(means,costs,x_data,y_labels):
    #
    nviews = x_data.shape[1] # features num is view
    diff = np.expand_dims(x_data,axis=1) - means[None,:,:] # (nsamp, class, nviews)
    diff_sq = np.square(diff)
    # now, for all combinations
    rel_est = np.zeros(2**nviews - 1)
    cost_vec = np.zeros(2**nviews - 1)
    for b in range(1,2**nviews):
        b_offset = b -1
        sel_idx = np.array([int(bit) for bit in f"{b:0{nviews}b}"]).astype(bool) # NOTE: make it index selector
        # (make sure 3 dim even single view selection)
        subset_diff = np.sum(diff_sq[:,:,sel_idx],axis=2)
        # output shape (nsamp, class)
        tmp_pred = np.argmin(subset_diff,axis=1)
        rel_est[b_offset] = np.mean(tmp_pred == y_labels)
        cost_vec[b_offset] = np.sum(costs[sel_idx])
    return rel_est, cost_vec

def sim_unbiased(xdata,costs,num_rounds,budget_ratio,means,y_labels,seed):
    rng = np.random.default_rng(seed=seed)
    nclasses = means.shape[0] # (nc, xdim)
    nviews = means.shape[1]
    # init weights
    regular_scale = 1.0
    m_est = rng.normal(size=(nclasses,nviews))
    covs_est = np.stack([regular_scale * np.eye(nviews) for _ in range(nclasses)],axis=0)
    ncnt_mat = np.ones((nclasses,nviews,nviews)) # for stability
    # reliability
    reward_est = np.ones(2**nviews - 1)/nclasses
    reward_cnt = np.ones(2**nviews - 1)
    # subset mappings
    subset_maps_cache = [[] for _ in range(2**nviews - 1)] # on the run
    # costs
    costs_vec = np.ones(2**nviews - 1)
    for b in range(1,2**nviews):
      subset = np.array([int(bit) for bit in f"{b:0{nviews}b}"]).astype("bool")
      costs_vec[b-1] = np.sum(costs[subset])

    # optimization
    free_indices = np.array([idx for idx in range(nviews) if costs[idx] == 0])

    omd_lambda = 0
    lambda_max = 10.0
    alpha_ucb = 2.0
    step_size= 0.05
    remain_budget = num_rounds * budget_ratio # initialization

    # recording
    record_acc = np.zeros((num_rounds))

    #approx_reliability, costs_vec = reliability_learn(means,costs,xdata,y_labels)
    #opt_prob = lp_oracle(approx_reliability,costs_vec,budget_ratio,num_rounds)
    # # clean up the sparse prob,
    #clean_opt_pair = [( p , np.array([int(bit) for bit in f"{idx:0{nviews}b}"]).astype(bool) ) for idx,p in enumerate(opt_prob) if p > 0]
    #opt_prob, opt_subsets = zip(*clean_opt_pair)
    #opt_prob = np.array(opt_prob)
    #opt_subsets = np.array(opt_subsets)

    # testing caching miss rate
    #debug_cache_hit = 0
    #debug_cache_missed = 0

    for t in range(num_rounds):
        if t < nclasses:
            is_init = True
            subset = np.ones((nviews)).astype("bool")
        else:
            is_init = False
            #subset = rng.choice(opt_subsets,p=opt_prob)
            # use the ucb for reliability
            bonus_conf = alpha_ucb * np.sqrt(np.log(t+1) / reward_cnt)
            max_bang_per_buck = reward_est - omd_lambda * costs_vec + bonus_conf
            max_idx = np.argmax(max_bang_per_buck)
            # NOTE: shift the index offset by 1 due to the no view case
            subset = np.array([int(bit) for bit in f"{max_idx+1:0{nviews}b}"]).astype("bool")

        # checking the budget
        inst_cost = np.sum(costs[subset])
        if remain_budget >= inst_cost:
            remain_budget -= inst_cost
        else:
            break # system stop when budget is exhausted
            # override with free indices
            subset = np.zeros((nviews)).astype("bool")
            subset[free_indices] = True
        # observe
        tmp_cnt = np.diagonal(ncnt_mat,axis1=1,axis2=2) # (nc, nview)
        mean_est =  m_est / tmp_cnt # (nc,nviews)
        if not is_init:
            x_obs = xdata[t,subset]
            # estimate the statistics based on partial counters and partial sums
            sel_mean = mean_est[:,subset] # (nc, sel)
            y_pred = pred_linear_cla(x_obs,sel_mean)
        else:
            y_pred = rng.integers(nclasses)
        # for reward and update (external)
        y_true = y_labels[t]
        reward = y_pred == y_true
        # update weights
        x_masked = xdata[t] * subset.astype("float64")
        m_est[y_true] += x_masked
        covs_est[y_true] += np.outer(x_masked,x_masked)
        ncnt_mat[y_true] += np.outer(subset.astype("int"),subset.astype("int"))
        # subset update
        bmask_idx = bool_array_2_int(subset) -1 # offset 1
        # update the reliability of the current subset
        # Update the running average for the chosen subset's reliability
        reward_est[bmask_idx] = (reward_est[bmask_idx] * reward_cnt[bmask_idx] + reward) / (reward_cnt[bmask_idx] + 1)
        reward_cnt[bmask_idx] += 1
        # cache check
        if np.sum(subset) == 1:
            subsubsets = []
        elif len(subset_maps_cache[bmask_idx])!=0:
            subsubsets = subset_maps_cache[bmask_idx]
        else:
            # compute once
            subsubsets = enumerate_subsets_fast(subset)
            subset_maps_cache[bmask_idx] = subsubsets

        # go through all subsets and make pseudo prediction again
        for sub in subsubsets:
            # if correct, update the reliability accordingly
            sub_mean = mean_est[:,sub]
            y_sub_pred = pred_linear_cla(xdata[t,sub],sub_mean)
            sb_index = bool_array_2_int(sub) - 1 # offset 1
            # Update the running average for the sub-subset's reliability
            sub_reward = (y_sub_pred == y_true)
            reward_est[sb_index] = (reward_est[sb_index] * reward_cnt[sb_index] + sub_reward) / (reward_cnt[sb_index] + 1)
            reward_cnt[sb_index] += 1
        raw_lambda = omd_lambda + step_size * (inst_cost - budget_ratio)
        omd_lambda = max(0, min(lambda_max, raw_lambda))
        # record results
        record_acc[t] = reward
    # collecting results
    # checking learned mean
    spending = budget_ratio * num_rounds - remain_budget
    reward = np.sum(record_acc)
    return {"reward":reward,"spending":spending,"avg_acc":np.mean(record_acc),"record_acc":record_acc}

def main(seed, num_views, num_classes, num_rounds, num_trials, budget_ratio, output):

    # recording
    result_pd = {"trial":[], "reward":[], "spending":[], "avg_acc":[]}
    # simulation
    time_start = time.perf_counter()
    for t in range(num_trials):
        rng = np.random.default_rng(seed=seed+t)
        x_data, means, y_labels = generate_data(num_rounds,
                                                num_classes,
                                                num_views,
                                                seed+t,
                                                snrdb=9)
        cost_vec = rng.random(size=(num_views,)) + 0.2 # avoid near zero cost
        # NOTE: free view disabled
        cost_vec /= np.sum(cost_vec) # NOTE: this could cause numerical error

        trial_start = time.perf_counter()
        sim_dict = sim_unbiased(x_data,cost_vec,num_rounds,budget_ratio,means,y_labels,seed)
        print(f"Trial {t + 1}/{num_trials}... (Time elapsed: {time.perf_counter() - trial_start:.3f}s)")
        # accumulating
        result_pd['trial'].append(t)
        result_pd['reward'].append(sim_dict['reward'])
        result_pd['spending'].append(sim_dict['spending'])
        result_pd['avg_acc'].append(sim_dict['avg_acc'])
    res_df = pd.DataFrame.from_dict(result_pd)
    res_df.to_csv(output,index=False)
    print(f"Simulation complete, total time elapsed: {time.perf_counter() - time_start:.3f}s")

# Define parameters directly for Colab execution
seed = 42
num_views = 10
num_classes = 4
num_rounds = 1000
num_trials = 20
budget_ratio = 0.7
output = "output_multiclass_gmm_supervised.csv"

# Call the main function with the defined parameters
main(seed, num_views, num_classes, num_rounds, num_trials, budget_ratio, output)


Trial 1/20... (Time elapsed: 5.565s)
Trial 2/20... (Time elapsed: 3.082s)
Trial 3/20... (Time elapsed: 4.005s)
Trial 4/20... (Time elapsed: 2.822s)
Trial 5/20... (Time elapsed: 3.336s)
Trial 6/20... (Time elapsed: 2.477s)
Trial 7/20... (Time elapsed: 3.803s)
Trial 8/20... (Time elapsed: 3.166s)
Trial 9/20... (Time elapsed: 3.595s)
Trial 10/20... (Time elapsed: 3.097s)
Trial 11/20... (Time elapsed: 3.169s)
Trial 12/20... (Time elapsed: 2.417s)
Trial 13/20... (Time elapsed: 2.675s)
Trial 14/20... (Time elapsed: 2.766s)
Trial 15/20... (Time elapsed: 3.913s)
Trial 16/20... (Time elapsed: 2.385s)
Trial 17/20... (Time elapsed: 2.981s)
Trial 18/20... (Time elapsed: 4.142s)
Trial 19/20... (Time elapsed: 2.852s)
Trial 20/20... (Time elapsed: 2.724s)
Simulation complete, total time elapsed: 65.028s
